# Clarity export loading

Microsoft Clarity exports analytics data as a semi-structured CSV containing multiple embedded report sections rather than a single flat table. The raw export is first loaded without a predefined header structure to preserve all section boundaries and metadata rows for downstream parsing. A msaking is also used to protect client's anonimity.

In [1]:
import pandas as pd

clarity_raw = pd.read_csv(
    '../../data/clarity_analysis_raw.csv',
    header=None
)

print(clarity_raw.head(20))

         0                              1            2    3    4
0      NaN                            NaN          NaN  NaN  NaN
1      NaN                            NaN          NaN  NaN  NaN
2   Metric                       Sessions          NaN  NaN  NaN
3      NaN                 Total sessions         1876  NaN  NaN
4      NaN                   Bot sessions          572  NaN  NaN
5      NaN                            NaN          NaN  NaN  NaN
6   Metric              Pages per session          NaN  NaN  NaN
7      NaN                        Average  1.225479744  NaN  NaN
8      NaN                            NaN          NaN  NaN  NaN
9   Metric                   Scroll depth          NaN  NaN  NaN
10     NaN                        Average        30.64  NaN  NaN
11     NaN                            NaN          NaN  NaN  NaN
12  Metric              Active time spent          NaN  NaN  NaN
13     NaN                    Active time           49  NaN  NaN
14     NaN               

## Automatic section detection

Unlike conventional analytics datasets, Clarity exports contain multiple report blocks inside a single CSV file. Section boundaries are detected programmatically using rows containing the keyword `Metric`, allowing the pipeline to automatically identify embedded report categories such as:

- Sessions
- Browsers
- Top pages
- Referrers
- Smart events
- JavaScript errors
- Performance overview, and few others.

This approach eliminates the need for manual spreadsheet cleaning and creates a reusable parsing workflow for future exports.

In [2]:
section_indices = clarity_raw[
    clarity_raw[0].astype(str).str.contains('Metric', na=False)
].index.tolist()

print(section_indices)

[2, 6, 9, 12, 16, 21, 27, 42, 56, 62, 76, 86, 92]


In [3]:
section_names = []

for idx in section_indices:
    section_name = clarity_raw.iloc[idx, 1]
    section_names.append(section_name)

print(section_names)

['Sessions', 'Pages per session', 'Scroll depth', 'Active time spent', 'Users overview', 'Insights', 'Browsers', 'Top pages', 'Smart events', 'Referrer', 'JavaScript errors', 'Performance overview', 'Bot traffic']


## Section extraction pipeline

After detecting section boundaries, each report block is sliced into its own dataframe dynamically. Empty rows are removed and each extracted section is stored inside a dictionary structure for modular downstream analysis.

This transforms the raw Clarity export into structured analytics tables that can be cleaned, queried, and integrated with GSC and GA4 datasets.

In [4]:
sections = {}

for i, start_idx in enumerate(section_indices):

    section_name = clarity_raw.iloc[start_idx, 1]

    # determine end of section
    if i < len(section_indices) - 1:
        end_idx = section_indices[i + 1]
    else:
        end_idx = len(clarity_raw)

    # slice section
    section_df = clarity_raw.iloc[start_idx + 1:end_idx].copy()

    # remove fully empty rows
    section_df = section_df.dropna(how='all')

    sections[section_name] = section_df.reset_index(drop=True)

print(sections.keys())

dict_keys(['Sessions', 'Pages per session', 'Scroll depth', 'Active time spent', 'Users overview', 'Insights', 'Browsers', 'Top pages', 'Smart events', 'Referrer', 'JavaScript errors', 'Performance overview', 'Bot traffic'])


## Top pages extraction and anonymization

The `Top pages` section was isolated and transformed into a clean dataframe containing page URLs and session counts. Because the repository is public, client-identifying domains are anonymized using a regex-based masking function that preserves URL structure while protecting organizational identity.

Example transformation:

```text
https://original-client-domain.com/blogs/example-page
→
https://client-domain.com/blogs/example-page

In [5]:
import re


CLIENT_ALIAS = 'client-domain.com'

def mask_domain(url):
    if pd.isna(url):
        return url

    return re.sub(
        r'https?://[^/]+',
        f'https://{CLIENT_ALIAS}',
        str(url)
    )

In [6]:
top_pages = sections['Top pages'].copy()

top_pages[1] = top_pages[1].apply(mask_domain)

print(top_pages.head())

     0                                                  1    2    3    4
0  NaN  https://client-domain.com/blogs/simple-compari...  473  NaN  NaN
1  NaN  https://client-domain.com/blogs/aws-in-2026-la...  370  NaN  NaN
2  NaN                         https://client-domain.com/  202  NaN  NaN
3  NaN  https://client-domain.com/blogs/amazon-bedrock...  168  NaN  NaN
4  NaN   https://client-domain.com/blogs/paypal-in-nepal/  133  NaN  NaN


In [7]:
top_pages = sections['Top pages'].copy()

# keep only relevant columns
top_pages = top_pages[[1, 2]]

# rename columns
top_pages.columns = ['page', 'sessions']

# mask client domain
top_pages['page'] = top_pages['page'].apply(mask_domain)

# convert sessions to numeric
top_pages['sessions'] = pd.to_numeric(
    top_pages['sessions'],
    errors='coerce'
)

# remove broken rows
top_pages = top_pages.dropna(subset=['page', 'sessions'])

print(top_pages.head())

                                                page  sessions
0  https://client-domain.com/blogs/simple-compari...       473
1  https://client-domain.com/blogs/aws-in-2026-la...       370
2                         https://client-domain.com/       202
3  https://client-domain.com/blogs/amazon-bedrock...       168
4   https://client-domain.com/blogs/paypal-in-nepal/       133
